In [1]:
import pandas as pd

# Rows 0-1 are metadata junk ("discrete" labels, blank row) — real data starts at row 2
df = pd.read_csv('../data/raw/kidney.csv')
df = df.iloc[2:].reset_index(drop=True)

print("Shape:", df.shape)
df.head()

Shape: (200, 29)


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,htn,dm,cad,appet,pe,ane,grf,stage,affected,age
0,0,0,1.019 - 1.021,1-Jan,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
1,0,0,1.009 - 1.011,< 0,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,≥ 227.944,s1,1,< 12
2,0,0,1.009 - 1.011,≥ 4,ckd,1,< 0,1,0,1,...,0,0,0,1,0,0,127.281 - 152.446,s1,1,< 12
3,1,1,1.009 - 1.011,3-Mar,ckd,0,< 0,0,0,0,...,0,0,0,0,0,0,127.281 - 152.446,s1,1,< 12
4,0,0,1.015 - 1.017,< 0,ckd,0,< 0,0,0,0,...,0,1,0,1,1,0,127.281 - 152.446,s1,1,20-Dec


In [2]:
print("class counts:")
print(df['class'].value_counts())

# 'affected' and 'stage' are derived directly from the target diagnosis —
# including them would be data leakage
print("\naffected vs class (should be a perfect 1:1 match):")
print(pd.crosstab(df['affected'], df['class']))

print("\nstage vs class:")
print(pd.crosstab(df['stage'], df['class']))

class counts:
class
ckd       128
notckd     72
Name: count, dtype: int64

affected vs class (should be a perfect 1:1 match):
class     ckd  notckd
affected             
0           0      72
1         128       0

stage vs class:
class  ckd  notckd
stage             
s1       9      45
s2      12      23
s3      31       0
s4      41       4
s5      35       0


In [3]:
df = df.drop(columns=['affected', 'stage'])

df['class'] = df['class'].map({'ckd': 1, 'notckd': 0})
df['class'].value_counts()

class
1    128
0     72
Name: count, dtype: int64

In [4]:
df['age'] = df['age'].replace({'20-Dec': '12 - 20'})
df['age'].unique()

array(['< 12', '12 - 20', '20 - 27', '27 - 35', '35 - 43', '43 - 51',
       '51 - 59', '59 - 66', '66 - 74', '≥ 74'], dtype=object)

In [5]:
import numpy as np
import re

def parse_bin(value):
    """Converts a bin-range string like '112 - 154', '< 48.1', or
    '≥ 227.944' into a single representative numeric value (midpoint,
    or the boundary itself for open-ended bins)."""
    if pd.isna(value):
        return np.nan
    value = str(value).strip()

    if value.startswith('<'):
        return float(value.replace('<', '').strip())
    if value.startswith('≥') or value.startswith('>='):
        return float(value.replace('≥', '').replace('>=', '').strip())
    if '-' in value:
        parts = value.split('-')
        try:
            lo, hi = float(parts[0].strip()), float(parts[1].strip())
            return (lo + hi) / 2
        except ValueError:
            return np.nan 
    try:
        return float(value)
    except ValueError:
        return np.nan

# True continuous-range columns (safe to midpoint-parse)
range_cols = ['sg', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'grf']

for col in range_cols:
    df[col] = df[col].apply(parse_bin)

df[range_cols].describe()

,sg,bgr,bu,sod,sc,pot,hemo,pcv,rbcc,wbcc,grf
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.00000,199.000000
mean,1.017450,157.465000,73.150750,137.387500,4.610750,7.662800,12.206750,37.751000,4.732875,8746.35000,70.309842
std,0.004835,73.376353,45.857702,6.377457,2.871741,3.413395,2.689822,7.725564,0.849932,2794.68954,53.426598
min,1.007000,112.000000,48.100000,118.000000,3.650000,7.310000,6.100000,17.900000,2.690000,4980.00000,26.617500
25%,1.016000,112.000000,48.100000,135.500000,3.650000,7.310000,10.650000,31.550000,4.165000,6170.00000,26.617500
50%,1.020000,133.000000,48.100000,135.500000,3.650000,7.310000,11.950000,39.350000,4.755000,8550.00000,39.200350
75%,1.020000,175.000000,67.150000,140.500000,3.650000,7.310000,14.550000,43.250000,5.345000,8550.00000,89.532000
max,1.023000,448.000000,352.900000,158.000000,28.850000,42.590000,16.500000,49.100000,7.410000,24020.00000,227.944000


In [6]:
print("NaNs introduced by parsing:")
print(df[range_cols].isnull().sum())

df[range_cols] = df[range_cols].fillna(df[range_cols].median())

NaNs introduced by parsing:
sg      0
bgr     0
bu      0
sod     0
sc      0
pot     0
hemo    0
pcv     0
rbcc    0
wbcc    0
grf     1
dtype: int64


In [7]:
al_order = ['< 0', '1-Jan', '2-Feb', '3-Mar', '≥ 4']
su_order = ['< 0', '2-Jan', '2-Feb', '4-Mar', '4-Apr', '≥ 4']

al_map = {label: rank for rank, label in enumerate(al_order)}
su_map = {label: rank for rank, label in enumerate(su_order)}

print("al values not covered by the map (should be empty):", set(df['al'].unique()) - set(al_map))
print("su values not covered by the map (should be empty):", set(df['su'].unique()) - set(su_map))

df['al'] = df['al'].map(al_map)
df['su'] = df['su'].map(su_map)

al values not covered by the map (should be empty): set()
su values not covered by the map (should be empty): set()


In [8]:
df['age'] = df['age'].apply(parse_bin)

binary_cols = ['bp (Diastolic)', 'bp limit', 'rbc', 'pc', 'pcc', 'ba',
               'htn', 'dm', 'cad', 'appet', 'pe', 'ane']
df[binary_cols] = df[binary_cols].astype(int)

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bp (Diastolic)  200 non-null    int64  
 1   bp limit        200 non-null    int64  
 2   sg              200 non-null    float64
 3   al              200 non-null    int64  
 4   class           200 non-null    int64  
 5   rbc             200 non-null    int64  
 6   su              200 non-null    int64  
 7   pc              200 non-null    int64  
 8   pcc             200 non-null    int64  
 9   ba              200 non-null    int64  
 10  bgr             200 non-null    float64
 11  bu              200 non-null    float64
 12  sod             200 non-null    float64
 13  sc              200 non-null    float64
 14  pot             200 non-null    float64
 15  hemo            200 non-null    float64
 16  pcv             200 non-null    float64
 17  rbcc            200 non-null    flo

In [9]:
print("Final missing values:")
print(df.isnull().sum().sum(), "total NaNs remaining")
df.head()

Final missing values:
0 total NaNs remaining


,bp (Diastolic),bp limit,sg,al,class,rbc,su,pc,pcc,ba,...,rbcc,wbcc,htn,dm,cad,appet,pe,ane,grf,age
0,0,0,1.020,1,1,0,0,0,0,0,...,4.755,8550.0,0,0,0,0,0,0,227.9440,12.0
1,0,0,1.010,0,1,0,0,0,0,0,...,4.755,13310.0,0,0,0,0,0,0,227.9440,12.0
2,0,0,1.010,4,1,1,0,1,0,1,...,4.755,15690.0,0,0,0,1,0,0,139.8635,12.0
3,1,1,1.010,3,1,0,0,0,0,0,...,4.755,8550.0,0,0,0,0,0,0,139.8635,12.0
4,0,0,1.016,0,1,0,0,0,0,0,...,5.345,8550.0,0,1,0,1,1,0,139.8635,16.0


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import json

X = df.drop(columns=['class'])
y = df['class']
feature_order = list(X.columns)
print("Features:", feature_order)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}")

Features: ['bp (Diastolic)', 'bp limit', 'sg', 'al', 'rbc', 'su', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sod', 'sc', 'pot', 'hemo', 'pcv', 'rbcc', 'wbcc', 'htn', 'dm', 'cad', 'appet', 'pe', 'ane', 'grf', 'age']
Train: (160, 26), Test: (40, 26)


In [11]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, recall_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)

xgb = XGBClassifier(eval_metric='logloss', random_state=42)
xgb.fit(X_train_scaled, y_train)
xgb_preds = xgb.predict(X_test_scaled)

print("--- Random Forest ---")
print("Accuracy:", accuracy_score(y_test, rf_preds))
print("Recall (class 1):", recall_score(y_test, rf_preds))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test_scaled)[:, 1]))
print(classification_report(y_test, rf_preds))

print("--- XGBoost ---")
print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("Recall (class 1):", recall_score(y_test, xgb_preds))
print("ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test_scaled)[:, 1]))
print(classification_report(y_test, xgb_preds))

--- Random Forest ---
Accuracy: 1.0
Recall (class 1): 1.0
ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        14
           1       1.00      1.00      1.00        26

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

--- XGBoost ---
Accuracy: 0.975
Recall (class 1): 0.9615384615384616
ROC-AUC: 0.9945054945054944
              precision    recall  f1-score   support

           0       0.93      1.00      0.97        14
           1       1.00      0.96      0.98        26

    accuracy                           0.97        40
   macro avg       0.97      0.98      0.97        40
weighted avg       0.98      0.97      0.98        40



In [12]:
import os, joblib

best_model = rf

output_dir = '../app/modules/kidney'
os.makedirs(output_dir, exist_ok=True)

joblib.dump(best_model, os.path.join(output_dir, 'model.pkl'))
joblib.dump(scaler, os.path.join(output_dir, 'scaler.pkl'))
with open(os.path.join(output_dir, 'feature_order.json'), 'w') as f:
    json.dump(feature_order, f)

print("Saved model.pkl, scaler.pkl, feature_order.json to app/modules/kidney/")

Saved model.pkl, scaler.pkl, feature_order.json to app/modules/kidney/


In [1]:
import sys
sys.path.append('..')

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from app.modules.kidney.train import load_and_clean

# Re-clean the raw data using the exact same pipeline train.py uses
df = load_and_clean()
X = df.drop(columns=['class'])
y = df['class']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = RandomForestClassifier(n_estimators=100, random_state=42)

# 5-fold cross-validation instead of a single 80/20 split
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_scaled, y, cv=cv, scoring='accuracy')

print("Accuracy per fold:", scores)
print("Mean accuracy:", scores.mean())
print("Std deviation:", scores.std())

Cleaned dataset: 200 rows, 27 columns
Class balance:
 class
1    128
0     72
Name: count, dtype: int64
Accuracy per fold: [1.    1.    1.    1.    0.975]
Mean accuracy: 0.9949999999999999
Std deviation: 0.010000000000000009
